# 🧬 Cultural SFT + nDNA Analysis: Llama-3.2-3B on Latin American Culture

**Research Objective:** Fine-tune Llama-3.2-3B-Instruct on English Wikipedia articles about Latin American Culture using LoRA with 8-bit quantization, then perform complete nDNA analysis comparing base vs fine-tuned model.

**Cultural Focus:** Latin American Culture (English Wikipedia corpus)
- Countries: Mexico, Brazil, Argentina, Colombia, Peru, Chile, Venezuela, Cuba, etc.
- Topics: Music (Salsa, Tango, Samba), Art (Frida Kahlo, Diego Rivera), Literature (Magical Realism)
- Heritage: Aztec, Maya, Inca civilizations, Indigenous cultures, Colonial history

**nDNA Components:**
- **Spectral Curvature (κₗ):** Measures latent manifold bending - how sharply internal trajectories bend
- **Thermodynamic Length (Lₗ):** Quantifies epistemic effort - how hard model works to adapt beliefs  
- **Belief Vector Field (‖vₗ(c)‖):** Encodes directional force from cultural priors

**Dataset:** [wikimedia/wikipedia](https://huggingface.co/datasets/wikimedia/wikipedia) (20231101.en) - filtered for Latin American cultural content

**Author:** Research Implementation | **Hardware:** Tesla P40 (20GB) / T4 GPU

In [1]:
# ============================================================================
# INSTALL REQUIRED PACKAGES (Run this ONCE, then restart kernel)
# ============================================================================
# For VS Code: Run this cell, then RESTART the kernel before running other cells

%pip install transformers accelerate peft datasets trl plotly scipy pandas matplotlib kaleido sentencepiece --quiet

print("✅ Packages installed! Please RESTART the kernel now, then run from Cell 2.")

Note: you may need to restart the kernel to use updated packages.
✅ Packages installed! Please RESTART the kernel now, then run from Cell 2.


In [ ]:
# !pip install -q transformers==4.44.0 huggingface_hub==0.24.0 accelerate>=0.27.0 peft>=0.10.0 \
#     bitsandbytes>=0.43.0 datasets>=2.18.0 trl>=0.8.0 \
#     plotly>=5.18.0 scipy pandas matplotlib kaleido sentencepiece

## 1️⃣ Setup and Dependencies

In [1]:
import os, gc, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy import linalg
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, HTML

warnings.filterwarnings('ignore')

# Fixed random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Check GPU
print(f"🔧 PyTorch: {torch.__version__}")
print(f"🎮 CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📊 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ============================================================================
# HUGGINGFACE LOGIN (Required for gated models like Llama)
# ============================================================================
# Get your token from: https://huggingface.co/settings/tokens
# Make sure you have accepted the Llama license at: https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct

from huggingface_hub import login

# Option 1: Use environment variable (recommended for Colab)
import os
hf_token = os.environ.get("HF_TOKEN", None)

# Option 2: Uncomment and paste your token directly (less secure but works)
# hf_token = "hf_your_token_here"

if hf_token:
    login(token=hf_token)
    print("✅ Logged in to HuggingFace Hub")
else:
    print("⚠️ No HF_TOKEN found. Trying notebook login...")
    try:
        # This will prompt for token in Colab
        login()
        print("✅ Logged in to HuggingFace Hub")
    except Exception as e:
        print(f"⚠️ HuggingFace login skipped: {str(e)[:50]}")
        print("   If you get access errors, set HF_TOKEN or run: huggingface-cli login")

🔧 PyTorch: 2.6.0+cu124
🎮 CUDA Available: True
📊 GPU: NVIDIA H100 80GB HBM3
💾 VRAM: 84.9 GB
⚠️ No HF_TOKEN found. Trying notebook login...


✅ Logged in to HuggingFace Hub


In [2]:
from google.colab import drive
import os

# Mount Google Drive
print("Mounting Google Drive...")
try:
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully.")
except Exception as e:
    print(f"❌ Error mounting Google Drive: {e}")

Mounting Google Drive...
❌ Error mounting Google Drive: Mountpoint must not already contain files


## 2️⃣ Configure Local Storage Paths (Persistent Across Sessions)

In [2]:
# ============================================================================
# CONFIGURE OUTPUT PATHS - LOCAL PERSISTENT STORAGE
# ============================================================================
# These paths point to YOUR LOCAL MACHINE's directory structure.
# Models saved here will persist across Colab sessions and can be used:
#   1. For future nDNA analysis runs (set TRAIN_NEW_MODEL=False)
#   2. For inference in other projects
#   3. For transfer to other machines

# ============================================================================
# 📁 LOCAL DIRECTORY PATHS - CHANGE THESE TO YOUR PREFERRED LOCATION
# ============================================================================
# Option A: If running on LOCAL MACHINE (Windows/Linux/Mac)
# LOCAL_BASE_DIR = "E:/nDNA/30Nov2025/FinetunedModels"  # Windows example
# LOCAL_BASE_DIR = "/home/user/nDNA/FinetunedModels"    # Linux example

# Option B: If running on COLAB and want to persist to Google Drive
# Uncomment the next 3 lines to use Google Drive instead:
# from google.colab import drive
# drive.mount('/content/drive')
# LOCAL_BASE_DIR = "/content/drive/MyDrive/nDNA_LatinAmerican/FinetunedModels"

# ============================================================================
# 🔧 SET YOUR LOCAL DIRECTORY HERE
# ============================================================================
LOCAL_BASE_DIR = "/nDNA/"  # ← CHANGE THIS PATH AS NEEDED

# Derived paths
OUTPUT_DIR = LOCAL_BASE_DIR
RESULTS_DIR = f"{LOCAL_BASE_DIR}/Results"
LORA_ADAPTER_PATH = f"{OUTPUT_DIR}/latin_american_lora_adapter"
MERGED_MODEL_PATH = f"{OUTPUT_DIR}/latin_american_llama_merged"

# Create directories if they don't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Model configuration
BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"

# ============================================================================
# 🔴 CRITICAL: SET THIS FLAG BASED ON YOUR SITUATION
# ============================================================================
# Set to True  → FIRST TIME: Will fine-tune the model and save it locally
# Set to False → LATER SESSIONS: Will load existing fine-tuned model (skip training)
#
# WORKFLOW:
#   Session 1: TRAIN_NEW_MODEL = True  → Trains and saves model to local directory
#   Session 2+: TRAIN_NEW_MODEL = False → Loads saved model for inference/analysis

TRAIN_NEW_MODEL = True  # ← Change to False after first successful training!

# ============================================================================

# Check if fine-tuned model already exists in local storage
lora_exists = os.path.exists(LORA_ADAPTER_PATH) and os.path.exists(f"{LORA_ADAPTER_PATH}/adapter_config.json")
merged_exists = os.path.exists(MERGED_MODEL_PATH) and os.path.exists(f"{MERGED_MODEL_PATH}/config.json")

print("=" * 70)
print("🔧 LOCAL STORAGE CONFIGURATION")
print("=" * 70)
print(f"📁 Base directory: {LOCAL_BASE_DIR}")
print(f"📁 Output directory: {OUTPUT_DIR}")
print(f"📁 Results directory: {RESULTS_DIR}")
print(f"🔗 Base model: {BASE_MODEL}")
print(f"\n📊 Existing Model Status (Local Storage):")
print(f"   LoRA adapter exists: {'✅ YES' if lora_exists else '❌ NO'}")
if lora_exists:
    print(f"      Path: {LORA_ADAPTER_PATH}")
print(f"   Merged model exists: {'✅ YES' if merged_exists else '❌ NO'}")
if merged_exists:
    print(f"      Path: {MERGED_MODEL_PATH}")
print(f"\n🎯 Mode: {'🔥 TRAINING MODE (will fine-tune and save locally)' if TRAIN_NEW_MODEL else '📥 INFERENCE MODE (loading from local storage)'}")

if not TRAIN_NEW_MODEL and not lora_exists:
    print("\n⚠️ WARNING: TRAIN_NEW_MODEL=False but no saved model found locally!")
    print("   Either set TRAIN_NEW_MODEL=True or check your local path.")
elif not TRAIN_NEW_MODEL and lora_exists:
    print("\n✅ Saved model found! Will load from local storage.")
print("=" * 70)

🔧 LOCAL STORAGE CONFIGURATION
📁 Base directory: /nDNA/
📁 Output directory: /nDNA/
📁 Results directory: /nDNA//Results
🔗 Base model: meta-llama/Llama-3.2-3B-Instruct

📊 Existing Model Status (Local Storage):
   LoRA adapter exists: ❌ NO
   Merged model exists: ❌ NO

🎯 Mode: 🔥 TRAINING MODE (will fine-tune and save locally)


## 3️⃣ Load English Wikipedia Dataset - Latin American Culture Articles
**Note:** This cell loads English Wikipedia and filters for Latin American cultural content. In INFERENCE MODE, we only need a small sample for nDNA analysis prompts.

In [3]:
# ============================================================================
# LOAD ENGLISH WIKIPEDIA - LATIN AMERICAN CULTURE ARTICLES
# ============================================================================
from datasets import load_dataset, Dataset
import re

print("=" * 70)
print("📚 LOADING ENGLISH WIKIPEDIA - LATIN AMERICAN CULTURE DATASET")
print("=" * 70)

# ============================================================================
# LATIN AMERICAN CULTURE KEYWORDS FOR FILTERING
# ============================================================================
# Comprehensive list covering countries, cultures, music, art, food, history

LATIN_AMERICA_KEYWORDS = [
    # Countries and Nationalities
    "mexico", "mexican", "brazil", "brazilian", "argentina", "argentine", "argentinian",
    "colombia", "colombian", "peru", "peruvian", "chile", "chilean",
    "venezuela", "venezuelan", "cuba", "cuban", "ecuador", "ecuadorian",
    "bolivia", "bolivian", "paraguay", "paraguayan", "uruguay", "uruguayan",
    "guatemala", "guatemalan", "costa rica", "costa rican", "panama", "panamanian",
    "honduras", "honduran", "nicaragua", "nicaraguan", "el salvador", "salvadoran",
    "dominican republic", "dominican", "puerto rico", "puerto rican",
    "caribbean", "latin america", "latino", "latina", "latinx", "hispanic",
    "central america", "south america",

    # Pre-Columbian Civilizations & Indigenous
    "aztec", "maya", "mayan", "inca", "incan", "olmec", "toltec", "zapotec",
    "mixtec", "teotihuacan", "chichen itza", "machu picchu", "nazca",
    "indigenous", "mestizo", "creole", "afro-latin", "afro-caribbean",
    "quechua", "nahuatl", "guarani", "aymara",

    # Music & Dance
    "salsa", "tango", "samba", "bossa nova", "reggaeton", "cumbia", "bachata",
    "mariachi", "ranchera", "bolero", "merengue", "rumba", "cha-cha", "mambo",
    "capoeira", "tropicalia", "latin jazz", "tejano", "norteño", "vallenato",
    "son cubano", "trova", "nueva trova", "lambada", "forró",

    # Art & Artists
    "frida kahlo", "diego rivera", "david alfaro siqueiros", "jose clemente orozco",
    "fernando botero", "wilfredo lam", "rufino tamayo", "roberto matta",
    "muralism", "muralismo", "latin american art",

    # Literature & Authors
    "gabriel garcia marquez", "pablo neruda", "jorge luis borges", "octavio paz",
    "mario vargas llosa", "julio cortazar", "isabel allende", "carlos fuentes",
    "magical realism", "realismo magico", "boom latinoamericano",
    "latin american literature", "one hundred years of solitude", "cien años de soledad",

    # Food & Cuisine
    "taco", "burrito", "empanada", "ceviche", "feijoada", "arepa", "tamale", "tamales",
    "mole", "guacamole", "tortilla", "churro", "dulce de leche", "pupusa",
    "chimichurri", "asado", "pisco", "tequila", "mezcal", "caipirinha",
    "latin american cuisine", "mexican food", "brazilian cuisine",

    # Festivals & Traditions
    "carnival", "carnaval", "dia de los muertos", "day of the dead",
    "cinco de mayo", "quinceañera", "posada", "semana santa",
    "lucha libre", "telenovela", "novela",

    # Geography & Cities
    "amazon", "andes", "patagonia", "yucatan", "oaxaca", "chiapas",
    "rio de janeiro", "sao paulo", "buenos aires", "mexico city", "ciudad de mexico",
    "havana", "lima", "bogota", "santiago", "caracas", "montevideo",
    "copacabana", "ipanema", "acapulco", "cancun",

    # Historical Terms
    "conquistador", "colonial", "pre-columbian", "mesoamerica", "mesoamerican",
    "spanish conquest", "portuguese colonization", "latin american independence",
    "simon bolivar", "jose de san martin", "pancho villa", "emiliano zapata",
    "fidel castro", "che guevara", "eva peron", "juan peron"
]

def contains_latin_american_content(text, title):
    """
    Check if Wikipedia article contains Latin American cultural content.
    Searches in title + first 5000 characters of text for efficiency.
    """
    if not text or not title:
        return False
    # Combine title and beginning of text for keyword search
    combined = (title.lower() + " " + text[:5000].lower())
    return any(keyword in combined for keyword in LATIN_AMERICA_KEYWORDS)

# ============================================================================
# LOAD AND FILTER WIKIPEDIA DATASET
# ============================================================================

# Samples configuration based on mode
if TRAIN_NEW_MODEL:
    TARGET_SAMPLES = 20000   # Target samples for training(#$50000)
    STREAM_LIMIT = 50000    # How many articles to scan (Wikipedia has 6M+ articles)(#500000)
else:
    TARGET_SAMPLES = 1000    # Small sample for inference mode
    STREAM_LIMIT = 10000

print(f"🎯 Mode: {'TRAINING' if TRAIN_NEW_MODEL else 'INFERENCE'}")
print(f"📊 Target samples: {TARGET_SAMPLES:,}")
print(f"🔍 Will scan up to {STREAM_LIMIT:,} Wikipedia articles")
print()

# Load Wikipedia English dataset with streaming (memory efficient)
print("📥 Loading English Wikipedia (streaming mode)...")
wiki_stream = load_dataset(
    "wikimedia/wikipedia",
    "20231101.en",
    split="train",
    streaming=True,
    trust_remote_code=True
)

# Filter for Latin American content
print("🔎 Filtering for Latin American cultural content...")
filtered_articles = []
scanned = 0
matches_found = 0

for article in wiki_stream:
    scanned += 1

    title = article.get("title", "")
    text = article.get("text", "")

    if contains_latin_american_content(text, title):
        # Keep only the text field (consistent with training format)
        filtered_articles.append({"text": text})
        matches_found += 1

        if matches_found % 1000 == 0:
            print(f"   Found {matches_found:,} articles... (scanned {scanned:,})")

    if matches_found >= TARGET_SAMPLES or scanned >= STREAM_LIMIT:
        break

print(f"\n✅ Filtering complete!")
print(f"   Scanned: {scanned:,} articles")
print(f"   Matched: {matches_found:,} Latin American culture articles")

# Convert to HuggingFace Dataset
combined_dataset = Dataset.from_list(filtered_articles)

print(f"\n{'='*70}")
print(f"📊 DATASET STATISTICS:")
print(f"   Total samples: {len(combined_dataset):,}")
print(f"   Language: English")
print(f"   Source: Wikipedia (20231101.en)")
print(f"   Filter: Latin American Culture keywords ({len(LATIN_AMERICA_KEYWORDS)} keywords)")
print(f"   Column names: {combined_dataset.column_names}")
print(f"{'='*70}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'wikimedia/wikipedia' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


📚 LOADING ENGLISH WIKIPEDIA - LATIN AMERICAN CULTURE DATASET
🎯 Mode: TRAINING
📊 Target samples: 50,000
🔍 Will scan up to 500,000 Wikipedia articles

📥 Loading English Wikipedia (streaming mode)...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

🔎 Filtering for Latin American cultural content...
   Found 1,000 articles... (scanned 3,401)
   Found 2,000 articles... (scanned 8,669)
   Found 3,000 articles... (scanned 14,226)
   Found 4,000 articles... (scanned 19,491)
   Found 5,000 articles... (scanned 24,388)
   Found 6,000 articles... (scanned 29,169)
   Found 7,000 articles... (scanned 34,369)
   Found 8,000 articles... (scanned 39,254)
   Found 9,000 articles... (scanned 44,626)
   Found 10,000 articles... (scanned 49,648)
   Found 11,000 articles... (scanned 54,910)
   Found 12,000 articles... (scanned 59,806)
   Found 13,000 articles... (scanned 65,522)
   Found 14,000 articles... (scanned 70,434)
   Found 15,000 articles... (scanned 75,849)
   Found 16,000 articles... (scanned 80,823)
   Found 17,000 articles... (scanned 86,271)
   Found 18,000 articles... (scanned 92,068)
   Found 19,000 articles... (scanned 97,841)
   Found 20,000 articles... (scanned 103,513)
   Found 21,000 articles... (scanned 109,292)
   Found 22,0

## 4️⃣ Preprocess and Split Dataset (90% Train / 10% Validation)
**Note:** This prepares data for training. In inference mode, we still need val_dataset for nDNA analysis prompts.

In [4]:
# ============================================================================
# PREPROCESS AND SPLIT DATASET
# ============================================================================

def preprocess_text(examples):
    """Minimal preprocessing - strip whitespace, filter empty"""
    texts = []
    for text in examples["text"]:
        if text and isinstance(text, str):
            cleaned = text.strip()
            if len(cleaned) > 50:  # Keep only meaningful text
                texts.append(cleaned)
        else:
            texts.append("")
    return {"text": texts}

# Apply preprocessing
print("🔄 Preprocessing dataset...")
combined_dataset = combined_dataset.map(
    preprocess_text,
    batched=True,
    remove_columns=[c for c in combined_dataset.column_names if c != "text"]
)

# Filter empty texts
combined_dataset = combined_dataset.filter(lambda x: len(x["text"]) > 50)
print(f"   After filtering: {len(combined_dataset):,} samples")

# Split 90% train, 10% validation
split = combined_dataset.train_test_split(test_size=0.1, seed=SEED)
train_dataset = split["train"]
val_dataset = split["test"]

print(f"\n📊 Final Split:")
print(f"   Train: {len(train_dataset):,} samples (90%)")
print(f"   Validation: {len(val_dataset):,} samples (10%)")

🔄 Preprocessing dataset...


Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/50000 [00:00<?, ? examples/s]

   After filtering: 50,000 samples

📊 Final Split:
   Train: 45,000 samples (90%)
   Validation: 5,000 samples (10%)


## 5️⃣ Load Base Model with 8-bit Quantization
**⏭️ TRAINING MODE ONLY** - This cell loads model for fine-tuning. Skip to Cell 10 in inference mode.

In [5]:
# ============================================================================
# LOAD BASE MODEL (TRAINING MODE ONLY)
# ============================================================================
# NOTE: Using FP16 instead of 8-bit quantization for Windows compatibility
# bitsandbytes has limited Windows support, FP16 works reliably

if not TRAIN_NEW_MODEL:
    print("⏭️ SKIPPING: Inference mode - will load fine-tuned model directly in Cell 10")
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM

    print(f"📥 Loading {BASE_MODEL} with FP16 precision...")
    print("   (Using FP16 for Windows compatibility - bitsandbytes not required)")

    # Load tokenizer with fallback for compatibility
    try:
        tokenizer = AutoTokenizer.from_pretrained(
            BASE_MODEL,
            trust_remote_code=True,
            use_fast=True,
        )
    except Exception as e:
        print(f"   ⚠️ Fast tokenizer failed, trying slow tokenizer: {str(e)[:50]}")
        tokenizer = AutoTokenizer.from_pretrained(
            BASE_MODEL,
            trust_remote_code=True,
            use_fast=False,
        )

    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    print(f"   ✅ Tokenizer loaded: {type(tokenizer).__name__}")

    # Load model with FP16 (no quantization - Windows compatible)
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.float16,  # FP16 for memory efficiency
        low_cpu_mem_usage=True,     # Reduce CPU memory during loading
    )
    model.config.use_cache = False

    print(f"✅ Model loaded successfully!")
    print(f"   Precision: FP16 (float16)")
    print(f"   Parameters: {model.num_parameters():,}")
    print(f"   Layers: {model.config.num_hidden_layers}")
    print(f"   VRAM Usage: ~{model.num_parameters() * 2 / 1e9:.1f} GB (FP16)")

📥 Loading meta-llama/Llama-3.2-3B-Instruct with FP16 precision...
   (Using FP16 for Windows compatibility - bitsandbytes not required)


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

   ✅ Tokenizer loaded: PreTrainedTokenizerFast


...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

✅ Model loaded successfully!
   Precision: FP16 (float16)
   Parameters: 3,212,749,824
   Layers: 28
   VRAM Usage: ~6.4 GB (FP16)


In [6]:
!pip freeze

absl-py==2.3.1
anyio==4.10.0
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
arrow==1.3.0
asttokens==3.0.0
async-lru==2.0.5
attrs==25.3.0
autobahn==24.4.2
Automat==25.4.16
babel==2.17.0
bash_kernel==0.10.0
beautifulsoup4==4.13.5
bleach==6.2.0
blinker==1.4
certifi==2025.8.3
cffi==2.0.0
charset-normalizer==3.4.3
click==8.2.1
comm==0.2.3
conda-pack==0.8.1
constantly==23.10.4
cryptography==45.0.7
dbus-python==1.2.18
debugpy==1.8.16
decorator==5.2.1
defusedxml==0.7.1
distro==1.7.0
exceptiongroup==1.3.0
executing==2.2.1
fastjsonschema==2.21.2
filetype==1.2.0
fqdn==1.5.1
grpcio==1.74.0
h11==0.16.0
httpcore==1.0.9
httplib2==0.20.2
httpx==0.28.1
humanize==4.13.0
hyperlink==21.0.0
idna==3.10
importlib-metadata==4.6.4
incremental==24.7.2
iniconfig==2.1.0
iotop==0.6
ipykernel==6.30.1
ipython==8.37.0
ipywidgets==8.1.7
isoduration==20.11.0
iterable-io==1.0.0
jedi==0.19.2
jeepney==0.7.1
Jinja2==3.1.6
json5==0.12.1
jsonpointer==3.0.0
jsonschema==4.25.1
jsonschema-specifications==2025.9.1
jupyter==1.1

## 6️⃣ Configure LoRA and PEFT
**⏭️ TRAINING MODE ONLY** - Skip in inference mode.

In [7]:
# ============================================================================
# CONFIGURE LORA FOR PARAMETER-EFFICIENT FINE-TUNING (TRAINING MODE ONLY)
# ============================================================================
if not TRAIN_NEW_MODEL:
    print("⏭️ SKIPPING: Inference mode - no LoRA configuration needed")
else:
    from peft import LoraConfig, get_peft_model

    # LoRA configuration - targeting key attention modules
    lora_config = LoraConfig(
        r=16,                          # LoRA rank
        lora_alpha=32,                 # Scaling factor
        target_modules=[               # Target attention layers
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )

    # Apply LoRA to the FP16 model
    model = get_peft_model(model, lora_config)

    # Enable gradient checkpointing for memory efficiency
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()

    # Print trainable parameters
    trainable_params, total_params = 0, 0
    for _, param in model.named_parameters():
        total_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()

    print(f"✅ LoRA Applied!")
    print(f"   Trainable: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")
    print(f"   Total: {total_params:,}")

✅ LoRA Applied!
   Trainable: 24,313,856 (0.75%)
   Total: 3,237,063,680


## 7️⃣ Training Configuration and SFTTrainer Setup
**⏭️ TRAINING MODE ONLY** - Skip in inference mode.

In [8]:
# ============================================================================
# TRAINING CONFIGURATION (TRAINING MODE ONLY)
# ============================================================================

from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

if not TRAIN_NEW_MODEL:
    print("⏭️ SKIPPING: Inference mode - no training configuration needed")
else:
    # SFTConfig for newer trl versions (replaces TrainingArguments for SFT)
    sft_config = SFTConfig(
        output_dir="./sft_cultural_output",
        num_train_epochs=1,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=8,      # Effective batch size = 16
        learning_rate=2e-4,
        weight_decay=0.01,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        logging_steps=20,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        fp16=True,
        optim="adamw_torch",               # Windows compatible (no bitsandbytes)
        max_grad_norm=0.3,
        seed=SEED,
        report_to="none",
        gradient_checkpointing=True,
        # SFT-specific settings
        packing=True,
        dataset_text_field="text",
    )

    # Initialize SFT Trainer (newer API)
    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,        # New API: 'processing_class' instead of 'tokenizer'
    )

    print("✅ SFTTrainer configured!")
    print(f"   Batch size: {sft_config.per_device_train_batch_size}")
    print(f"   Gradient accumulation: {sft_config.gradient_accumulation_steps}")
    print(f"   Effective batch: {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}")
    print(f"   Optimizer: {sft_config.optim}")

Padding-free training is enabled, but the attention implementation is not set to a supported flash attention variant. Padding-free training flattens batches into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn, kernels-community/flash-attn3, kernels-community/vllm-flash-attn3. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation` in the model configuration to one of these supported options or verify that your attention mechanism can handle flattened sequences.
You are using packing, but the attention implementation is not set to a supported flash attention variant. Packing gathers multiple samples into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn, kernels-community/flash-attn3, kernels-community/vllm-fla

Adding EOS to train dataset:   0%|          | 0/45000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/45000 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/45000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Packing eval dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


✅ SFTTrainer configured!
   Batch size: 2
   Gradient accumulation: 8
   Effective batch: 16
   Optimizer: adamw_torch


## 8️⃣ Execute Fine-tuning
**⏭️ TRAINING MODE ONLY** - Skip in inference mode.

In [ ]:
# ============================================================================
# EXECUTE FINE-TUNING (TRAINING MODE ONLY)
# ============================================================================
if not TRAIN_NEW_MODEL:
    print("⏭️ SKIPPING: Inference mode - using pre-trained model")
    train_loss_history = []
    eval_loss_history = []
else:
    print("🚀 Starting Cultural SFT Training...")
    print("=" * 60)

    # Train the model
    train_result = trainer.train()

    # Extract training metrics from log history
    train_loss_history = []
    eval_loss_history = []

    for log in trainer.state.log_history:
        if "loss" in log and "eval_loss" not in log:
            train_loss_history.append(log["loss"])
        if "eval_loss" in log:
            eval_loss_history.append(log["eval_loss"])

    print("\n" + "=" * 60)
    print("✅ Training Complete!")
    print(f"   Final Train Loss: {train_result.training_loss:.4f}")
    print(f"   Total Steps: {trainer.state.global_step}")
    print(f"   Train Loss Records: {len(train_loss_history)}")
    print(f"   Eval Loss Records: {len(eval_loss_history)}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


🚀 Starting Cultural SFT Training...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
200,2.148500,2.153325,2.137183,3252723.000000,0.532922
400,2.140500,2.138918,2.181558,6505315.000000,0.535425
600,2.121000,2.129918,2.131447,9758380.000000,0.536824
800,2.125800,2.123022,2.133573,13011461.000000,0.538036
1000,2.111600,2.118042,2.141567,16261916.000000,0.538672
1200,2.107900,2.113495,2.136201,19514719.000000,0.539533
1400,2.097000,2.110020,2.124175,22768595.000000,0.540016


## 9️⃣ Save Fine-tuned Model to Local Directory
**⏭️ TRAINING MODE ONLY** - Skip in inference mode (model already saved locally).

After this cell completes, your fine-tuned model will be saved to:
- `{LOCAL_BASE_DIR}/latin_american_lora_adapter/` - LoRA weights (always saved)
- `{LOCAL_BASE_DIR}/latin_american_llama_merged/` - Full merged model (if possible)

In [ ]:
# ============================================================================
# SAVE FINE-TUNED MODEL TO LOCAL DIRECTORY (TRAINING MODE ONLY)
# ============================================================================
if not TRAIN_NEW_MODEL:
    print("⏭️ SKIPPING: Inference mode - will load model from local storage")
    MERGED_MODEL_AVAILABLE = os.path.exists(MERGED_MODEL_PATH) and os.path.exists(f"{MERGED_MODEL_PATH}/config.json")
    print(f"   LoRA adapter path: {LORA_ADAPTER_PATH}")
    print(f"   Merged model available: {MERGED_MODEL_AVAILABLE}")
else:
    print("=" * 70)
    print("💾 SAVING FINE-TUNED MODEL TO LOCAL DIRECTORY")
    print("=" * 70)
    print(f"📁 Target directory: {OUTPUT_DIR}")

    # Save LoRA adapter (this always works, even with 8-bit models)
    print("\n1️⃣ Saving LoRA adapter...")
    model.save_pretrained(LORA_ADAPTER_PATH)
    tokenizer.save_pretrained(LORA_ADAPTER_PATH)
    print(f"   ✅ LoRA adapter saved to: {LORA_ADAPTER_PATH}")

    # List saved files
    import glob
    saved_files = glob.glob(f"{LORA_ADAPTER_PATH}/*")
    print(f"   📄 Files saved: {len(saved_files)}")
    for f in saved_files[:5]:
        print(f"      - {os.path.basename(f)}")
    if len(saved_files) > 5:
        print(f"      ... and {len(saved_files) - 5} more files")

    # Try to save merged model (may fail with 8-bit quantization)
    print("\n2️⃣ Attempting to merge and save full model...")
    try:
        merged_model = model.merge_and_unload()
        merged_model.save_pretrained(MERGED_MODEL_PATH, safe_serialization=True)
        tokenizer.save_pretrained(MERGED_MODEL_PATH)
        print(f"   ✅ Merged model saved to: {MERGED_MODEL_PATH}")
        MERGED_MODEL_AVAILABLE = True
    except Exception as e:
        print(f"   ⚠️ Merge failed (normal for 8-bit quantized models)")
        print(f"   📌 Reason: {str(e)[:80]}")
        print("   📌 This is expected behavior - will use LoRA adapter for inference")
        MERGED_MODEL_AVAILABLE = False

    # Save training history
    print("\n3️⃣ Saving training history...")
    max_len = max(len(train_loss_history), len(eval_loss_history)) if train_loss_history or eval_loss_history else 0
    if max_len > 0:
        train_padded = train_loss_history + [None] * (max_len - len(train_loss_history))
        eval_padded = eval_loss_history + [None] * (max_len - len(eval_loss_history))
        history_path = f"{RESULTS_DIR}/training_history.csv"
        pd.DataFrame({"train_loss": train_padded, "eval_loss": eval_padded}).to_csv(history_path, index=False)
        print(f"   ✅ Training history saved to: {history_path}")

    # Free training memory
    del trainer
    gc.collect()
    torch.cuda.empty_cache()

    print("\n" + "=" * 70)
    print("✅ MODEL SAVED SUCCESSFULLY TO LOCAL DIRECTORY!")
    print("=" * 70)
    print(f"\n📁 SAVED LOCATIONS:")
    print(f"   LoRA Adapter: {LORA_ADAPTER_PATH}")
    if MERGED_MODEL_AVAILABLE:
        print(f"   Merged Model: {MERGED_MODEL_PATH}")
    else:
        print(f"   Merged Model: Not available (using LoRA adapter)")
    print(f"   Results: {RESULTS_DIR}")

    print(f"\n💡 FOR FUTURE SESSIONS:")
    print(f"   1. Set TRAIN_NEW_MODEL = False in Cell 2")
    print(f"   2. Run Cells 1-4, then skip to Cell 10")
    print(f"   3. All nDNA analysis will use your saved model!")

    print(f"\n📦 TO USE IN OTHER PROJECTS:")
    print(f"   Copy this path: {LORA_ADAPTER_PATH}")
    print(f"   See the 'Quick Reference' section at the end of this notebook")
    print("=" * 70)

## 🔟 Load Both Models for nDNA Comparison
**✅ ALWAYS RUN** - This loads both base and fine-tuned models for analysis.

| Mode | What Happens |
|------|--------------|
| **Training Mode** (`TRAIN_NEW_MODEL=True`) | Loads fresh models after training completes |
| **Inference Mode** (`TRAIN_NEW_MODEL=False`) | Loads saved fine-tuned model from **local directory** |

**Local Model Path:** `{LOCAL_BASE_DIR}/latin_american_lora_adapter/`

In [ ]:
# ============================================================================
# LOAD BOTH MODELS FOR nDNA COMPARISON
# ============================================================================
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

print("📥 Loading models for nDNA analysis...")

# Clear any existing models from memory
for var_name in ['model', 'trainer', 'merged_model', 'base_model', 'finetuned_model']:
    if var_name in dir():
        exec(f"del {var_name}")
gc.collect()
torch.cuda.empty_cache()

# Define quantization config (needed for loading)
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.float16,
    bnb_8bit_use_double_quant=True,
)

# Load tokenizer (use saved tokenizer if available, with fallback)
if os.path.exists(f"{LORA_ADAPTER_PATH}/tokenizer_config.json"):
    print("   Loading tokenizer from saved adapter...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(LORA_ADAPTER_PATH, trust_remote_code=True, use_fast=True)
    except:
        tokenizer = AutoTokenizer.from_pretrained(LORA_ADAPTER_PATH, trust_remote_code=True, use_fast=False)
else:
    print("   Loading tokenizer from base model...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True, use_fast=True)
    except:
        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True, use_fast=False)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(f"   ✅ Tokenizer loaded: {type(tokenizer).__name__}")

# ========== LOAD BASE MODEL (Fresh, Unmodified) ==========
print("\n🔵 Loading BASE model (unmodified)...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
base_model.eval()
print(f"   ✅ Base model loaded")

# ========== LOAD FINE-TUNED MODEL ==========
print("\n🔴 Loading FINE-TUNED model from local storage...")

# Check what's available
lora_exists = os.path.exists(LORA_ADAPTER_PATH) and os.path.exists(f"{LORA_ADAPTER_PATH}/adapter_config.json")
merged_exists = os.path.exists(MERGED_MODEL_PATH) and os.path.exists(f"{MERGED_MODEL_PATH}/config.json")

if not lora_exists and not merged_exists:
    raise ValueError(f"❌ No fine-tuned model found!\n   LoRA path: {LORA_ADAPTER_PATH}\n   Merged path: {MERGED_MODEL_PATH}\n   Please set TRAIN_NEW_MODEL=True and run training first.")

# Try merged model first, fall back to LoRA adapter
if merged_exists:
    try:
        print(f"   Loading merged model from: {MERGED_MODEL_PATH}")
        finetuned_model = AutoModelForCausalLM.from_pretrained(
            MERGED_MODEL_PATH,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.float16,
            trust_remote_code=True,
        )
        print(f"   ✅ Loaded merged model successfully")
    except Exception as e:
        print(f"   ⚠️ Failed to load merged model: {str(e)[:50]}")
        merged_exists = False

if not merged_exists and lora_exists:
    print(f"   Loading via LoRA adapter from: {LORA_ADAPTER_PATH}")
    # Load fresh base model
    finetuned_base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
    # Apply LoRA adapter
    finetuned_model = PeftModel.from_pretrained(
        finetuned_base,
        LORA_ADAPTER_PATH,
    )
    print(f"   ✅ Loaded LoRA adapter on base model")

finetuned_model.eval()
print(f"   ✅ Fine-tuned model ready for inference")

# Get number of layers
NUM_LAYERS = base_model.config.num_hidden_layers
print(f"\n✅ Both models loaded successfully!")
print(f"   Number of layers: {NUM_LAYERS}")
print(f"   Hidden size: {base_model.config.hidden_size}")
print(f"   Source: {'Merged model' if merged_exists else 'LoRA adapter'} from local storage")

## 1️⃣1️⃣ Define Cultural and Generic Prompts

In [ ]:
# ============================================================================
# EXTRACT RAW PROMPTS FROM THE ACTUAL DATASET FOR nDNA ANALYSIS
# ============================================================================
# IMPORTANT: Using REAL text samples from English Wikipedia Latin American
# cultural articles, NOT synthetic prompts!

print("📝 Extracting raw text samples from Latin American cultural dataset for nDNA analysis...")

# Get samples from the actual training dataset
# These are REAL cultural texts from English Wikipedia about Latin America

# Number of samples for nDNA analysis (more samples = better statistical validity)
N_NDNA_SAMPLES = 100  # Use 100 samples for robust analysis

# Sample from the training dataset - these are REAL cultural texts from Wikipedia
dataset_samples = train_dataset.shuffle(seed=SEED).select(range(min(N_NDNA_SAMPLES, len(train_dataset))))

# Extract raw text prompts from dataset
RAW_CULTURAL_PROMPTS = []
for sample in dataset_samples:
    text = sample["text"]
    # Use first 200 characters as prompt (sufficient for hidden state extraction)
    # This captures the Latin American cultural essence without being too long
    if len(text) > 50:
        prompt = text[:200].strip()
        RAW_CULTURAL_PROMPTS.append(prompt)

print(f"   ✅ Extracted {len(RAW_CULTURAL_PROMPTS)} raw cultural prompts from English Wikipedia (Latin America)")

# Also get some samples from validation set for diversity
val_samples = val_dataset.shuffle(seed=SEED).select(range(min(20, len(val_dataset))))
for sample in val_samples:
    text = sample["text"]
    if len(text) > 50:
        prompt = text[:200].strip()
        RAW_CULTURAL_PROMPTS.append(prompt)

print(f"   ✅ Total cultural prompts: {len(RAW_CULTURAL_PROMPTS)}")

# Display a few examples of actual prompts being used
print("\n📋 Sample prompts from dataset (first 3):")
for i, p in enumerate(RAW_CULTURAL_PROMPTS[:3]):
    print(f"   {i+1}. {p[:100]}...")

# ALL_PROMPTS for nDNA analysis will use ONLY real dataset texts
ALL_PROMPTS = RAW_CULTURAL_PROMPTS
print(f"\n✅ Total prompts for nDNA analysis: {len(ALL_PROMPTS)} (all from real Latin American cultural dataset)")

## 1️⃣2️⃣ nDNA Analysis Framework - Core Implementation

In [ ]:
# ============================================================================
# nDNA ANALYSIS FRAMEWORK - CORE IMPLEMENTATION
# Based on: https://pragyaai.github.io/ndna/llm/ndna/
# ============================================================================

class nDNAAnalyzer:
    """
    Neural DNA (nDNA) Analyzer for Foundation Models

    Computes three key metrics per layer:
    1. Spectral Curvature (κₗ): Measures latent manifold bending
    2. Thermodynamic Length (Lₗ): Quantifies epistemic effort
    3. Belief Vector Field Norm (‖vₗ(c)‖): Directional cultural force

    Reference: https://pragyaai.github.io/ndna/llm/ndna/
    """

    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.num_layers = model.config.num_hidden_layers
        self.hidden_size = model.config.hidden_size
        # Get device from model parameters
        self.device = next(model.parameters()).device

    def extract_hidden_states(self, text):
        """Extract hidden states from all layers for a given text"""
        try:
            # Tokenize with proper handling
            inputs = self.tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=256,
                padding=True
            )
            # Move to model's device
            inputs = {k: v.to(self.device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = self.model(
                    **inputs,
                    output_hidden_states=True,
                    return_dict=True
                )

            # hidden_states is a tuple of (num_layers + 1) tensors
            # Each tensor shape: [batch_size, seq_len, hidden_size]
            hidden_states = outputs.hidden_states

            # Stack and average over sequence length
            # Result: [num_layers + 1, hidden_size]
            stacked = torch.stack([h.mean(dim=1).squeeze(0) for h in hidden_states], dim=0)

            return stacked.float().cpu().numpy()

        except Exception as e:
            print(f"   ⚠️ Error extracting hidden states: {str(e)[:50]}")
            # Return zeros as fallback
            return np.zeros((self.num_layers + 1, self.hidden_size))

    def compute_spectral_curvature(self, hidden_states):
        """
        Compute Spectral Curvature: κₗ = ‖Δ²hₗ‖ = ‖hₗ₊₁ - 2hₗ + hₗ₋₁‖

        This is the discrete second derivative (Laplacian) of the trajectory.
        High curvature = sharp semantic pivot, belief compression

        Formula from nDNA paper: κₗ := ‖hₗ₊₁ - 2hₗ + hₗ₋₁‖
        """
        n_layers = len(hidden_states)
        kappa = np.zeros(n_layers)

        for l in range(1, n_layers - 1):
            # Second-order difference (discrete Laplacian along depth)
            delta2 = hidden_states[l+1] - 2*hidden_states[l] + hidden_states[l-1]
            kappa[l] = np.linalg.norm(delta2)

        # Edge layers: use nearest computed value
        kappa[0] = kappa[1] if n_layers > 1 else 0
        kappa[-1] = kappa[-2] if n_layers > 1 else 0

        return kappa

    def compute_thermodynamic_length(self, hidden_states):
        """
        Compute Thermodynamic Length: Lₗ = ‖hₗ - hₗ₋₁‖

        Measures the "distance traveled" in latent space between layers.
        High Lₗ = internal effort, belief transition

        This is related to the Fisher information metric integral.
        """
        n_layers = len(hidden_states)
        L = np.zeros(n_layers)

        for l in range(1, n_layers):
            delta = hidden_states[l] - hidden_states[l-1]
            L[l] = np.linalg.norm(delta)

        return L

    def compute_belief_vector_field(self, hidden_states):
        """
        Compute Belief Vector Field Norm: ‖vₗ‖

        The belief vector field represents the direction and magnitude
        of semantic force at each layer. We approximate it as the
        first derivative (velocity) of the trajectory.

        High norm = strong directional pressure in latent space
        """
        n_layers = len(hidden_states)
        v_norms = np.zeros(n_layers)

        for l in range(n_layers - 1):
            v = hidden_states[l+1] - hidden_states[l]
            v_norms[l] = np.linalg.norm(v)

        # Last layer: copy from previous
        v_norms[-1] = v_norms[-2] if n_layers > 1 else 0

        return v_norms

    def compute_alignment_cosine(self, hidden_states):
        """
        Compute cos(θ) between consecutive velocity vectors.

        High alignment = smooth, consistent flow direction
        Low alignment = direction changes, semantic reorganization
        """
        n_layers = len(hidden_states)
        alignments = np.zeros(n_layers)

        for l in range(1, n_layers - 1):
            v_prev = hidden_states[l] - hidden_states[l-1]
            v_next = hidden_states[l+1] - hidden_states[l]

            norm_prev = np.linalg.norm(v_prev)
            norm_next = np.linalg.norm(v_next)

            if norm_prev > 1e-8 and norm_next > 1e-8:
                alignments[l] = np.dot(v_prev, v_next) / (norm_prev * norm_next)

        alignments[0] = alignments[1] if n_layers > 1 else 0
        alignments[-1] = alignments[-2] if n_layers > 1 else 0

        return alignments

    def compute_ndna_score(self, kappa, L, v_norm, omega=None):
        """
        Compute unified nDNA score: nDNA = Σ ωₗ · κₗ · Lₗ · ‖vₗ‖

        This multiplicative form highlights layers where:
        - Path bends sharply (high κ)
        - Significant effort is expended (high L)
        - Strong directional force exists (high ‖v‖)
        """
        n_layers = len(kappa)

        if omega is None:
            # Weight upper layers more (where cultural adaptation is strongest)
            # Based on nDNA finding that layers [20,30] in 30-layer models are key
            omega = np.linspace(0.5, 1.5, n_layers)

        ndna_per_layer = omega * kappa * L * v_norm
        total_ndna = np.sum(ndna_per_layer)

        return ndna_per_layer, total_ndna

    def analyze_prompts(self, prompts, batch_progress=True):
        """
        Run full nDNA analysis on a list of prompts.
        Returns averaged metrics across all prompts.
        """
        all_kappa = []
        all_L = []
        all_v = []
        all_align = []

        n_prompts = len(prompts)
        print(f"🔬 Analyzing {n_prompts} prompts...")

        for i, prompt in enumerate(prompts):
            if batch_progress and (i % 20 == 0 or i == n_prompts - 1):
                print(f"   Progress: {i+1}/{n_prompts} ({100*(i+1)/n_prompts:.1f}%)")

            # Extract hidden states for this prompt
            hidden = self.extract_hidden_states(prompt)

            # Compute all nDNA metrics
            kappa = self.compute_spectral_curvature(hidden)
            L = self.compute_thermodynamic_length(hidden)
            v = self.compute_belief_vector_field(hidden)
            align = self.compute_alignment_cosine(hidden)

            all_kappa.append(kappa)
            all_L.append(L)
            all_v.append(v)
            all_align.append(align)

        # Convert to numpy arrays
        all_kappa = np.array(all_kappa)
        all_L = np.array(all_L)
        all_v = np.array(all_v)
        all_align = np.array(all_align)

        # Compute statistics across all prompts
        metrics = {
            "spectral_curvature": np.mean(all_kappa, axis=0),
            "thermodynamic_length": np.mean(all_L, axis=0),
            "belief_vector_norm": np.mean(all_v, axis=0),
            "alignment_cosine": np.mean(all_align, axis=0),
            "spectral_curvature_std": np.std(all_kappa, axis=0),
            "thermodynamic_length_std": np.std(all_L, axis=0),
            "belief_vector_norm_std": np.std(all_v, axis=0),
            # Also store per-prompt data for detailed analysis
            "all_kappa": all_kappa,
            "all_L": all_L,
            "all_v": all_v,
        }

        # Compute unified nDNA score
        ndna_layers, ndna_total = self.compute_ndna_score(
            metrics["spectral_curvature"],
            metrics["thermodynamic_length"],
            metrics["belief_vector_norm"]
        )
        metrics["ndna_per_layer"] = ndna_layers
        metrics["ndna_total"] = ndna_total

        print(f"   ✅ Analysis complete!")

        return metrics

print("✅ nDNA Analyzer class defined successfully!")

## 1️⃣3️⃣ Run nDNA Analysis on Base and Fine-tuned Models

In [ ]:
# ============================================================================
# RUN nDNA ANALYSIS ON BOTH MODELS
# ============================================================================
print("=" * 70)
print("🧬 RUNNING nDNA ANALYSIS")
print("=" * 70)

# Analyze BASE model
print("\n📊 Analyzing BASE model...")
base_analyzer = nDNAAnalyzer(base_model, tokenizer)
base_metrics = base_analyzer.analyze_prompts(ALL_PROMPTS)
print(f"   ✅ Base nDNA Total Score: {base_metrics['ndna_total']:.4f}")

# Clear some memory
torch.cuda.empty_cache()

# Analyze FINE-TUNED model
print("\n📊 Analyzing FINE-TUNED model...")
ft_analyzer = nDNAAnalyzer(finetuned_model, tokenizer)
ft_metrics = ft_analyzer.analyze_prompts(ALL_PROMPTS)
print(f"   ✅ Fine-tuned nDNA Total Score: {ft_metrics['ndna_total']:.4f}")

# Compute deltas (fine-tuned - base)
deltas = {
    "spectral_curvature_delta": ft_metrics["spectral_curvature"] - base_metrics["spectral_curvature"],
    "thermodynamic_length_delta": ft_metrics["thermodynamic_length"] - base_metrics["thermodynamic_length"],
    "belief_vector_norm_delta": ft_metrics["belief_vector_norm"] - base_metrics["belief_vector_norm"],
    "ndna_per_layer_delta": ft_metrics["ndna_per_layer"] - base_metrics["ndna_per_layer"],
}

print("\n✅ nDNA Analysis Complete!")
print(f"   nDNA Change: {ft_metrics['ndna_total'] - base_metrics['ndna_total']:.4f}")

## 1️⃣4️⃣ Save nDNA Metrics to CSV

In [ ]:
# ============================================================================
# SAVE nDNA METRICS TO CSV
# ============================================================================
layers = list(range(NUM_LAYERS + 1))

# Create comprehensive DataFrame
ndna_df = pd.DataFrame({
    "layer": layers,
    # Base model metrics
    "base_spectral_curvature": base_metrics["spectral_curvature"],
    "base_thermodynamic_length": base_metrics["thermodynamic_length"],
    "base_belief_vector_norm": base_metrics["belief_vector_norm"],
    "base_alignment_cosine": base_metrics["alignment_cosine"],
    "base_ndna_per_layer": base_metrics["ndna_per_layer"],
    # Fine-tuned model metrics
    "ft_spectral_curvature": ft_metrics["spectral_curvature"],
    "ft_thermodynamic_length": ft_metrics["thermodynamic_length"],
    "ft_belief_vector_norm": ft_metrics["belief_vector_norm"],
    "ft_alignment_cosine": ft_metrics["alignment_cosine"],
    "ft_ndna_per_layer": ft_metrics["ndna_per_layer"],
    # Deltas
    "delta_spectral_curvature": deltas["spectral_curvature_delta"],
    "delta_thermodynamic_length": deltas["thermodynamic_length_delta"],
    "delta_belief_vector_norm": deltas["belief_vector_norm_delta"],
    "delta_ndna": deltas["ndna_per_layer_delta"],
})

# Save to CSV
ndna_df.to_csv(f"{RESULTS_DIR}/ndna_metrics_full.csv", index=False)
print(f"✅ Saved: {RESULTS_DIR}/ndna_metrics_full.csv")

# Also create a summary table
summary_df = pd.DataFrame({
    "metric": ["spectral_curvature", "thermodynamic_length", "belief_vector_norm", "ndna_total"],
    "base_mean": [
        np.mean(base_metrics["spectral_curvature"]),
        np.mean(base_metrics["thermodynamic_length"]),
        np.mean(base_metrics["belief_vector_norm"]),
        base_metrics["ndna_total"]
    ],
    "ft_mean": [
        np.mean(ft_metrics["spectral_curvature"]),
        np.mean(ft_metrics["thermodynamic_length"]),
        np.mean(ft_metrics["belief_vector_norm"]),
        ft_metrics["ndna_total"]
    ],
    "delta_mean": [
        np.mean(deltas["spectral_curvature_delta"]),
        np.mean(deltas["thermodynamic_length_delta"]),
        np.mean(deltas["belief_vector_norm_delta"]),
        ft_metrics["ndna_total"] - base_metrics["ndna_total"]
    ]
})
summary_df.to_csv(f"{RESULTS_DIR}/ndna_summary_table.csv", index=False)
print(f"✅ Saved: {RESULTS_DIR}/ndna_summary_table.csv")

# Display the summary
display(summary_df)

## 1️⃣5️⃣ Interactive 3D Plotly Surface - Spectral Curvature

In [ ]:
# ============================================================================
# INTERACTIVE 3D PLOTS - SPECTRAL CURVATURE
# ============================================================================
import plotly.io as pio

# Set renderer for Colab compatibility
try:
    import google.colab
    pio.renderers.default = "colab"
except:
    pio.renderers.default = "notebook"

# Layer indices
layers = list(range(len(base_metrics["spectral_curvature"])))

# Create 3D scatter plot showing both models' trajectories
fig_kappa_3d = go.Figure()

# Base model - Spectral Curvature by layer
fig_kappa_3d.add_trace(go.Scatter3d(
    x=layers,
    y=[0] * len(layers),  # y=0 for base model
    z=base_metrics["spectral_curvature"],
    mode='lines+markers',
    marker=dict(size=6, color='blue', symbol='circle'),
    line=dict(color='blue', width=4),
    name='Base Model (κ)'
))

# Fine-tuned model - Spectral Curvature by layer
fig_kappa_3d.add_trace(go.Scatter3d(
    x=layers,
    y=[1] * len(layers),  # y=1 for fine-tuned
    z=ft_metrics["spectral_curvature"],
    mode='lines+markers',
    marker=dict(size=6, color='red', symbol='diamond'),
    line=dict(color='red', width=4),
    name='Fine-tuned Model (κ)'
))

# Add vertical lines connecting corresponding layers (showing delta)
for l in range(0, len(layers), 3):  # Every 3rd layer for clarity
    fig_kappa_3d.add_trace(go.Scatter3d(
        x=[l, l],
        y=[0, 1],
        z=[base_metrics["spectral_curvature"][l], ft_metrics["spectral_curvature"][l]],
        mode='lines',
        line=dict(color='gray', width=2, dash='dash'),
        showlegend=False,
        hoverinfo='skip'
    ))

fig_kappa_3d.update_layout(
    title=dict(
        text="🔬 Spectral Curvature (κₗ) - 3D View: Base vs Fine-tuned",
        font=dict(size=18)
    ),
    scene=dict(
        xaxis_title="Layer (ℓ)",
        yaxis_title="Model",
        zaxis_title="Spectral Curvature (κₗ)",
        yaxis=dict(
            tickmode='array',
            ticktext=['Base', 'Fine-tuned'],
            tickvals=[0, 1]
        ),
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
    ),
    width=950, height=650,
    legend=dict(x=0.02, y=0.98),
    margin=dict(l=0, r=0, t=60, b=0)
)

fig_kappa_3d.show()

# Save as interactive HTML
fig_kappa_3d.write_html(f"{RESULTS_DIR}/spectral_curvature_3d.html")
print(f"✅ Saved: {RESULTS_DIR}/spectral_curvature_3d.html")

## 1️⃣6️⃣ Interactive 3D Surface - Thermodynamic Length

In [ ]:
# ============================================================================
# INTERACTIVE 3D PLOT - THERMODYNAMIC LENGTH
# ============================================================================
fig_thermo_3d = go.Figure()

# Base model
fig_thermo_3d.add_trace(go.Scatter3d(
    x=layers,
    y=[0] * len(layers),
    z=base_metrics["thermodynamic_length"],
    mode='lines+markers',
    marker=dict(size=6, color='green', symbol='circle'),
    line=dict(color='green', width=4),
    name='Base Model (L)'
))

# Fine-tuned model
fig_thermo_3d.add_trace(go.Scatter3d(
    x=layers,
    y=[1] * len(layers),
    z=ft_metrics["thermodynamic_length"],
    mode='lines+markers',
    marker=dict(size=6, color='orange', symbol='diamond'),
    line=dict(color='orange', width=4),
    name='Fine-tuned Model (L)'
))

# Connection lines
for l in range(0, len(layers), 3):
    fig_thermo_3d.add_trace(go.Scatter3d(
        x=[l, l], y=[0, 1],
        z=[base_metrics["thermodynamic_length"][l], ft_metrics["thermodynamic_length"][l]],
        mode='lines', line=dict(color='gray', width=2, dash='dash'),
        showlegend=False, hoverinfo='skip'
    ))

fig_thermo_3d.update_layout(
    title=dict(text="🔥 Thermodynamic Length (Lₗ) - 3D View: Base vs Fine-tuned", font=dict(size=18)),
    scene=dict(
        xaxis_title="Layer (ℓ)",
        yaxis_title="Model",
        zaxis_title="Thermodynamic Length (Lₗ)",
        yaxis=dict(tickmode='array', ticktext=['Base', 'Fine-tuned'], tickvals=[0, 1]),
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
    ),
    width=950, height=650,
    legend=dict(x=0.02, y=0.98)
)

fig_thermo_3d.show()
fig_thermo_3d.write_html(f"{RESULTS_DIR}/thermodynamic_length_3d.html")
print(f"✅ Saved: {RESULTS_DIR}/thermodynamic_length_3d.html")

## 1️⃣7️⃣ Interactive 3D Surface - Belief Vector Field

In [ ]:
# ============================================================================
# INTERACTIVE 3D PLOT - BELIEF VECTOR FIELD
# ============================================================================
fig_belief_3d = go.Figure()

# Base model
fig_belief_3d.add_trace(go.Scatter3d(
    x=layers,
    y=[0] * len(layers),
    z=base_metrics["belief_vector_norm"],
    mode='lines+markers',
    marker=dict(size=6, color='purple', symbol='circle'),
    line=dict(color='purple', width=4),
    name='Base Model (‖v‖)'
))

# Fine-tuned model
fig_belief_3d.add_trace(go.Scatter3d(
    x=layers,
    y=[1] * len(layers),
    z=ft_metrics["belief_vector_norm"],
    mode='lines+markers',
    marker=dict(size=6, color='magenta', symbol='diamond'),
    line=dict(color='magenta', width=4),
    name='Fine-tuned Model (‖v‖)'
))

# Connection lines
for l in range(0, len(layers), 3):
    fig_belief_3d.add_trace(go.Scatter3d(
        x=[l, l], y=[0, 1],
        z=[base_metrics["belief_vector_norm"][l], ft_metrics["belief_vector_norm"][l]],
        mode='lines', line=dict(color='gray', width=2, dash='dash'),
        showlegend=False, hoverinfo='skip'
    ))

fig_belief_3d.update_layout(
    title=dict(text="🎯 Belief Vector Field Norm (‖vₗ‖) - 3D View: Base vs Fine-tuned", font=dict(size=18)),
    scene=dict(
        xaxis_title="Layer (ℓ)",
        yaxis_title="Model",
        zaxis_title="Belief Field Norm (‖vₗ‖)",
        yaxis=dict(tickmode='array', ticktext=['Base', 'Fine-tuned'], tickvals=[0, 1]),
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
    ),
    width=950, height=650,
    legend=dict(x=0.02, y=0.98)
)

fig_belief_3d.show()
fig_belief_3d.write_html(f"{RESULTS_DIR}/belief_vector_field_3d.html")
print(f"✅ Saved: {RESULTS_DIR}/belief_vector_field_3d.html")

## 1️⃣8️⃣ Layer-by-Layer Comparison Plots (Base vs Fine-tuned)

In [ ]:
# ============================================================================
# LAYER-BY-LAYER COMPARISON PLOTS (INTERACTIVE)
# ============================================================================
fig_compare = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Spectral Curvature (κₗ)", "Thermodynamic Length (Lₗ)",
        "Belief Vector Norm (‖vₗ‖)", "nDNA Score per Layer"
    ],
    vertical_spacing=0.12, horizontal_spacing=0.1
)

# Spectral Curvature
fig_compare.add_trace(go.Scatter(x=layers, y=base_metrics["spectral_curvature"],
    mode='lines+markers', name='Base κ', line=dict(color='blue', width=2)), row=1, col=1)
fig_compare.add_trace(go.Scatter(x=layers, y=ft_metrics["spectral_curvature"],
    mode='lines+markers', name='FT κ', line=dict(color='red', width=2)), row=1, col=1)

# Thermodynamic Length
fig_compare.add_trace(go.Scatter(x=layers, y=base_metrics["thermodynamic_length"],
    mode='lines+markers', name='Base L', line=dict(color='green', width=2)), row=1, col=2)
fig_compare.add_trace(go.Scatter(x=layers, y=ft_metrics["thermodynamic_length"],
    mode='lines+markers', name='FT L', line=dict(color='orange', width=2)), row=1, col=2)

# Belief Vector Norm
fig_compare.add_trace(go.Scatter(x=layers, y=base_metrics["belief_vector_norm"],
    mode='lines+markers', name='Base ‖v‖', line=dict(color='purple', width=2)), row=2, col=1)
fig_compare.add_trace(go.Scatter(x=layers, y=ft_metrics["belief_vector_norm"],
    mode='lines+markers', name='FT ‖v‖', line=dict(color='magenta', width=2)), row=2, col=1)

# nDNA Score
fig_compare.add_trace(go.Scatter(x=layers, y=base_metrics["ndna_per_layer"],
    mode='lines+markers', name='Base nDNA', line=dict(color='teal', width=2)), row=2, col=2)
fig_compare.add_trace(go.Scatter(x=layers, y=ft_metrics["ndna_per_layer"],
    mode='lines+markers', name='FT nDNA', line=dict(color='crimson', width=2)), row=2, col=2)

fig_compare.update_layout(
    title=dict(text="📊 nDNA Metrics: Layer-by-Layer Comparison (Base vs Fine-tuned)", font=dict(size=18)),
    height=700, width=1100,
    showlegend=True, legend=dict(x=1.02, y=0.5)
)
fig_compare.update_xaxes(title_text="Layer (ℓ)")
fig_compare.show()
fig_compare.write_html(f"{RESULTS_DIR}/ndna_layer_comparison.html")
print(f"✅ Saved: {RESULTS_DIR}/ndna_layer_comparison.html")

## 1️⃣9️⃣ Combined nDNA Overlay Plot (Spectral + Thermo + Belief)

In [ ]:
# ============================================================================
# COMBINED nDNA OVERLAY PLOT
# ============================================================================
fig_combined = make_subplots(
    rows=2, cols=1,
    subplot_titles=["Base Model - nDNA Components", "Fine-tuned Model - nDNA Components"],
    vertical_spacing=0.15
)

# Normalize for overlay visualization
def normalize(arr):
    return (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)

# Base model
fig_combined.add_trace(go.Scatter(x=layers, y=normalize(base_metrics["spectral_curvature"]),
    mode='lines', name='κ (Curvature)', line=dict(color='blue', width=2), fill='tozeroy'), row=1, col=1)
fig_combined.add_trace(go.Scatter(x=layers, y=normalize(base_metrics["thermodynamic_length"]),
    mode='lines', name='L (Length)', line=dict(color='green', width=2)), row=1, col=1)
fig_combined.add_trace(go.Scatter(x=layers, y=normalize(base_metrics["belief_vector_norm"]),
    mode='lines', name='‖v‖ (Belief)', line=dict(color='purple', width=2)), row=1, col=1)

# Fine-tuned model
fig_combined.add_trace(go.Scatter(x=layers, y=normalize(ft_metrics["spectral_curvature"]),
    mode='lines', name='κ (FT)', line=dict(color='red', width=2, dash='dash'), fill='tozeroy'), row=2, col=1)
fig_combined.add_trace(go.Scatter(x=layers, y=normalize(ft_metrics["thermodynamic_length"]),
    mode='lines', name='L (FT)', line=dict(color='orange', width=2, dash='dash')), row=2, col=1)
fig_combined.add_trace(go.Scatter(x=layers, y=normalize(ft_metrics["belief_vector_norm"]),
    mode='lines', name='‖v‖ (FT)', line=dict(color='magenta', width=2, dash='dash')), row=2, col=1)

fig_combined.update_layout(
    title=dict(text="🧬 Combined nDNA Geometry: Normalized Overlay of All Components", font=dict(size=18)),
    height=700, width=1000, showlegend=True
)
fig_combined.update_xaxes(title_text="Layer (ℓ)")
fig_combined.update_yaxes(title_text="Normalized Value")
fig_combined.show()
fig_combined.write_html(f"{RESULTS_DIR}/ndna_combined_overlay.html")
print(f"✅ Saved: {RESULTS_DIR}/ndna_combined_overlay.html")

## 2️⃣0️⃣ Delta Analysis - What Changed After Fine-tuning?

In [ ]:
# ============================================================================
# DELTA ANALYSIS - CHANGES AFTER FINE-TUNING
# ============================================================================
fig_delta = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Δ Spectral Curvature", "Δ Thermodynamic Length",
        "Δ Belief Vector Norm", "Δ nDNA Score"
    ]
)

# Add delta plots with fill to show direction
for i, (name, delta, color) in enumerate([
    ("Δκ", deltas["spectral_curvature_delta"], 'blue'),
    ("ΔL", deltas["thermodynamic_length_delta"], 'green'),
    ("Δ‖v‖", deltas["belief_vector_norm_delta"], 'purple'),
    ("ΔnDNA", deltas["ndna_per_layer_delta"], 'crimson')
]):
    row, col = (i // 2) + 1, (i % 2) + 1
    fig_delta.add_trace(go.Bar(x=layers, y=delta, name=name,
        marker_color=[color if v >= 0 else 'gray' for v in delta]), row=row, col=col)
    fig_delta.add_hline(y=0, line_dash="dash", line_color="black", row=row, col=col)

fig_delta.update_layout(
    title=dict(text="📈 Delta Analysis: Changes in nDNA Metrics After Cultural Fine-tuning", font=dict(size=18)),
    height=650, width=1100, showlegend=True
)
fig_delta.update_xaxes(title_text="Layer (ℓ)")
fig_delta.show()
fig_delta.write_html(f"{RESULTS_DIR}/ndna_delta_analysis.html")
print(f"✅ Saved: {RESULTS_DIR}/ndna_delta_analysis.html")

## 2️⃣1️⃣ Aggregate Metrics Table and Summary Statistics

In [ ]:
# ============================================================================
# AGGREGATE METRICS TABLE
# ============================================================================
print("=" * 70)
print("📊 AGGREGATE nDNA METRICS SUMMARY")
print("=" * 70)

# Create summary table
summary_data = {
    "Metric": [
        "Spectral Curvature (κ) - Mean",
        "Spectral Curvature (κ) - Max",
        "Spectral Curvature (κ) - Std",
        "Thermodynamic Length (L) - Mean",
        "Thermodynamic Length (L) - Max",
        "Thermodynamic Length (L) - Std",
        "Belief Vector Norm (‖v‖) - Mean",
        "Belief Vector Norm (‖v‖) - Max",
        "Belief Vector Norm (‖v‖) - Std",
        "Total nDNA Score",
        "Peak nDNA Layer",
    ],
    "Base Model": [
        f"{np.mean(base_metrics['spectral_curvature']):.6f}",
        f"{np.max(base_metrics['spectral_curvature']):.6f}",
        f"{np.std(base_metrics['spectral_curvature']):.6f}",
        f"{np.mean(base_metrics['thermodynamic_length']):.6f}",
        f"{np.max(base_metrics['thermodynamic_length']):.6f}",
        f"{np.std(base_metrics['thermodynamic_length']):.6f}",
        f"{np.mean(base_metrics['belief_vector_norm']):.6f}",
        f"{np.max(base_metrics['belief_vector_norm']):.6f}",
        f"{np.std(base_metrics['belief_vector_norm']):.6f}",
        f"{base_metrics['ndna_total']:.6f}",
        f"Layer {np.argmax(base_metrics['ndna_per_layer'])}",
    ],
    "Fine-tuned Model": [
        f"{np.mean(ft_metrics['spectral_curvature']):.6f}",
        f"{np.max(ft_metrics['spectral_curvature']):.6f}",
        f"{np.std(ft_metrics['spectral_curvature']):.6f}",
        f"{np.mean(ft_metrics['thermodynamic_length']):.6f}",
        f"{np.max(ft_metrics['thermodynamic_length']):.6f}",
        f"{np.std(ft_metrics['thermodynamic_length']):.6f}",
        f"{np.mean(ft_metrics['belief_vector_norm']):.6f}",
        f"{np.max(ft_metrics['belief_vector_norm']):.6f}",
        f"{np.std(ft_metrics['belief_vector_norm']):.6f}",
        f"{ft_metrics['ndna_total']:.6f}",
        f"Layer {np.argmax(ft_metrics['ndna_per_layer'])}",
    ],
    "Delta (FT - Base)": [
        f"{np.mean(deltas['spectral_curvature_delta']):.6f}",
        f"{np.max(ft_metrics['spectral_curvature']) - np.max(base_metrics['spectral_curvature']):.6f}",
        f"{np.std(ft_metrics['spectral_curvature']) - np.std(base_metrics['spectral_curvature']):.6f}",
        f"{np.mean(deltas['thermodynamic_length_delta']):.6f}",
        f"{np.max(ft_metrics['thermodynamic_length']) - np.max(base_metrics['thermodynamic_length']):.6f}",
        f"{np.std(ft_metrics['thermodynamic_length']) - np.std(base_metrics['thermodynamic_length']):.6f}",
        f"{np.mean(deltas['belief_vector_norm_delta']):.6f}",
        f"{np.max(ft_metrics['belief_vector_norm']) - np.max(base_metrics['belief_vector_norm']):.6f}",
        f"{np.std(ft_metrics['belief_vector_norm']) - np.std(base_metrics['belief_vector_norm']):.6f}",
        f"{ft_metrics['ndna_total'] - base_metrics['ndna_total']:.6f}",
        "-",
    ],
}

summary_df = pd.DataFrame(summary_data)
display(summary_df.style.set_properties(**{'text-align': 'left'}))

# Save summary
summary_df.to_csv(f"{RESULTS_DIR}/ndna_summary_table.csv", index=False)
print(f"\n✅ Saved: {RESULTS_DIR}/ndna_summary_table.csv")

## 2️⃣2️⃣ Training Loss Curves

In [ ]:
# ============================================================================
# TRAINING LOSS CURVES
# ============================================================================
fig_loss = go.Figure()

# Load training history if available
try:
    history_df = pd.read_csv(f"{RESULTS_DIR}/training_history.csv")
    train_loss = history_df["train_loss"].dropna().tolist()
    eval_loss = history_df["eval_loss"].dropna().tolist()
except:
    train_loss = train_loss_history if 'train_loss_history' in dir() else []
    eval_loss = eval_loss_history if 'eval_loss_history' in dir() else []

if train_loss:
    fig_loss.add_trace(go.Scatter(
        y=train_loss, mode='lines+markers', name='Train Loss',
        line=dict(color='blue', width=2)
    ))

if eval_loss:
    eval_steps = np.linspace(0, len(train_loss)-1, len(eval_loss)).astype(int) if train_loss else range(len(eval_loss))
    fig_loss.add_trace(go.Scatter(
        x=eval_steps, y=eval_loss, mode='lines+markers', name='Eval Loss',
        line=dict(color='red', width=2)
    ))

fig_loss.update_layout(
    title=dict(text="📉 Training & Validation Loss Curves", font=dict(size=18)),
    xaxis_title="Training Steps",
    yaxis_title="Loss",
    height=450, width=900
)
fig_loss.show()
fig_loss.write_html(f"{RESULTS_DIR}/training_loss_curves.html")
print(f"✅ Saved: {RESULTS_DIR}/training_loss_curves.html")

## 2️⃣3️⃣ Inference Comparison: Base vs Fine-tuned

In [ ]:
# ============================================================================
# INFERENCE COMPARISON: BASE VS FINE-TUNED
# ============================================================================

def generate_response(model, prompt, max_new_tokens=100):
    """Generate response from model with proper device handling"""
    # Get model's device
    device = next(model.parameters()).device

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256,
        padding=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode and remove the input prompt from output
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Get only the generated part
    response = full_response[len(tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)):].strip()
    return response

# Use ACTUAL samples from the dataset for inference comparison
# Taking 10 samples from the actual cultural dataset
print("=" * 70)
print("🔍 INFERENCE COMPARISON: BASE vs FINE-TUNED")
print("=" * 70)

# Get fresh samples from validation set for inference testing
inference_samples = val_dataset.shuffle(seed=42).select(range(min(10, len(val_dataset))))
INFERENCE_PROMPTS = [s["text"][:150].strip() for s in inference_samples if len(s["text"]) > 50]

print(f"📝 Testing on {len(INFERENCE_PROMPTS)} real cultural text samples from dataset...\n")

inference_results = []
for i, prompt in enumerate(INFERENCE_PROMPTS):
    print(f"Sample {i+1}/{len(INFERENCE_PROMPTS)}")
    print(f"📌 Prompt: {prompt[:80]}...")

    try:
        base_response = generate_response(base_model, prompt)
        ft_response = generate_response(finetuned_model, prompt)

        inference_results.append({
            "prompt": prompt,
            "base_response": base_response[:200],
            "finetuned_response": ft_response[:200]
        })

        print(f"   🔵 Base:     {base_response[:80]}...")
        print(f"   🔴 FT:       {ft_response[:80]}...")
    except Exception as e:
        print(f"   ⚠️ Error: {str(e)[:50]}")
        inference_results.append({
            "prompt": prompt,
            "base_response": f"Error: {str(e)[:100]}",
            "finetuned_response": f"Error: {str(e)[:100]}"
        })
    print()

# Save inference results
inference_df = pd.DataFrame(inference_results)
inference_df.to_csv(f"{RESULTS_DIR}/inference_comparison.csv", index=False)
print(f"✅ Saved: {RESULTS_DIR}/inference_comparison.csv")

## 2️⃣4️⃣ Final 3D nDNA Signature Plot

In [ ]:
# ============================================================================
# FINAL 3D nDNA SIGNATURE PLOT - COMPREHENSIVE VIEW
# ============================================================================
fig_3d = go.Figure()

# Base model trajectory in (κ, L, ‖v‖) space
fig_3d.add_trace(go.Scatter3d(
    x=base_metrics["spectral_curvature"],
    y=base_metrics["thermodynamic_length"],
    z=base_metrics["belief_vector_norm"],
    mode='lines+markers',
    marker=dict(size=5, color=layers, colorscale='Blues', colorbar=dict(title="Layer", x=0.9)),
    line=dict(color='blue', width=3),
    name='Base Model',
    text=[f"Layer {l}" for l in layers],
    hovertemplate="Layer %{text}<br>κ=%{x:.4f}<br>L=%{y:.4f}<br>‖v‖=%{z:.4f}"
))

# Fine-tuned model trajectory
fig_3d.add_trace(go.Scatter3d(
    x=ft_metrics["spectral_curvature"],
    y=ft_metrics["thermodynamic_length"],
    z=ft_metrics["belief_vector_norm"],
    mode='lines+markers',
    marker=dict(size=5, color=layers, colorscale='Reds'),
    line=dict(color='red', width=3),
    name='Fine-tuned Model',
    text=[f"Layer {l}" for l in layers],
    hovertemplate="Layer %{text}<br>κ=%{x:.4f}<br>L=%{y:.4f}<br>‖v‖=%{z:.4f}"
))

fig_3d.update_layout(
    title=dict(
        text="🧬 3D nDNA Signature: Trajectory through (κ, L, ‖v‖) Space",
        font=dict(size=18)
    ),
    scene=dict(
        xaxis_title="Spectral Curvature (κₗ)",
        yaxis_title="Thermodynamic Length (Lₗ)",
        zaxis_title="Belief Vector Norm (‖vₗ‖)",
    ),
    width=1000, height=700,
    legend=dict(x=0.02, y=0.98)
)
fig_3d.show()
fig_3d.write_html(f"{RESULTS_DIR}/ndna_3d_signature.html")
print(f"✅ Saved: {RESULTS_DIR}/ndna_3d_signature.html")

## 2️⃣5️⃣ Research Summary and Key Findings

In [ ]:
# ============================================================================
# RESEARCH SUMMARY AND KEY FINDINGS
# ============================================================================
print("=" * 70)
print("🎓 RESEARCH SUMMARY: Latin American Cultural SFT + nDNA Analysis")
print("=" * 70)

# Calculate percentage change safely
def safe_pct_change(new_val, old_val):
    if abs(old_val) < 1e-10:
        return 0.0
    return 100 * (new_val - old_val) / abs(old_val)

kappa_pct = safe_pct_change(
    np.mean(ft_metrics['spectral_curvature']),
    np.mean(base_metrics['spectral_curvature'])
)

print(f"""
📊 EXPERIMENT CONFIGURATION:
   • Base Model: {BASE_MODEL}
   • Dataset: English Wikipedia - Latin American Culture
   • Source: wikimedia/wikipedia (20231101.en) filtered by cultural keywords
   • Training: LoRA (r=16, α=32) + 8-bit Quantization
   • Layers Analyzed: {NUM_LAYERS}
   • Analysis Prompts: {len(ALL_PROMPTS)} (from real cultural dataset)
   • Mode: {'Training + Analysis' if TRAIN_NEW_MODEL else 'Inference Only (loaded saved model)'}

🧬 nDNA ANALYSIS RESULTS:

   SPECTRAL CURVATURE (κₗ) - Measures latent manifold bending:
   • Base Mean: {np.mean(base_metrics['spectral_curvature']):.6f}
   • Fine-tuned Mean: {np.mean(ft_metrics['spectral_curvature']):.6f}
   • Change: {np.mean(deltas['spectral_curvature_delta']):.6f} ({'+' if kappa_pct > 0 else ''}{kappa_pct:.2f}%)
   • Peak Layer (FT): {np.argmax(ft_metrics['spectral_curvature'])}

   THERMODYNAMIC LENGTH (Lₗ) - Measures epistemic effort:
   • Base Mean: {np.mean(base_metrics['thermodynamic_length']):.6f}
   • Fine-tuned Mean: {np.mean(ft_metrics['thermodynamic_length']):.6f}
   • Change: {np.mean(deltas['thermodynamic_length_delta']):.6f}

   BELIEF VECTOR FIELD (‖vₗ‖) - Measures cultural steering force:
   • Base Mean: {np.mean(base_metrics['belief_vector_norm']):.6f}
   • Fine-tuned Mean: {np.mean(ft_metrics['belief_vector_norm']):.6f}
   • Change: {np.mean(deltas['belief_vector_norm_delta']):.6f}

   TOTAL nDNA SCORE:
   • Base: {base_metrics['ndna_total']:.6f}
   • Fine-tuned: {ft_metrics['ndna_total']:.6f}
   • Change: {ft_metrics['ndna_total'] - base_metrics['ndna_total']:.6f}

📁 OUTPUT FILES SAVED TO: {RESULTS_DIR}
   • ndna_metrics_full.csv - Full per-layer metrics
   • ndna_summary_table.csv - Aggregate statistics
   • inference_comparison.csv - Base vs FT responses
   • training_history.csv - Loss curves data
   • *.html - Interactive Plotly visualizations

💾 MODEL SAVED TO LOCAL STORAGE:
   • LoRA Adapter: {LORA_ADAPTER_PATH}
   • Merged Model: {MERGED_MODEL_PATH}

🔄 FOR FUTURE SESSIONS:
   1. Set TRAIN_NEW_MODEL = False in Cell 2
   2. Run Cells 1-4, then skip to Cell 10
   3. All nDNA analysis cells (10-25) will work with saved model

   You can also use the fine-tuned model for other purposes:

   from peft import PeftModel
   from transformers import AutoModelForCausalLM, AutoTokenizer

   model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", ...)
   model = PeftModel.from_pretrained(model, "{LORA_ADAPTER_PATH}")

🔬 KEY INSIGHTS:
   The nDNA analysis reveals how Latin American cultural fine-tuning reshapes
   the model's internal geometric fingerprint:

   • Spectral Curvature changes indicate where the model developed new
     semantic "bends" - potentially encoding Latin American cultural patterns
     (Aztec, Maya, Inca heritage; Salsa, Tango, Samba; Magical Realism)

   • Thermodynamic Length changes show where the model expends more/less
     epistemic effort after cultural adaptation to Latin American content

   • Belief Vector Field changes reveal where Latin American cultural priors
     exert directional influence on the latent manifold

   • Peak effects typically appear in upper layers (layers {NUM_LAYERS-10}-{NUM_LAYERS})
     where sociolinguistic and cultural priors exert strongest influence on output
""")

print("✅ Analysis Complete! All files saved successfully.")
print("=" * 70)

---
## 📚 References

1. **nDNA Framework**: [PragyaAI nDNA Documentation](https://pragyaai.github.io/ndna/llm/ndna/)
2. **Wikipedia Dataset**: [Wikimedia English Wikipedia](https://huggingface.co/datasets/wikimedia/wikipedia)
3. **Latin American Culture**: [Wikipedia - Culture of Latin America](https://en.wikipedia.org/wiki/Culture_of_Latin_America)
4. **Llama 3.2**: [Meta Llama 3.2 3B Instruct](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct)
5. **LoRA**: Hu et al., "LoRA: Low-Rank Adaptation of Large Language Models" (2021)
6. **Information Geometry**: Amari, "Information Geometry and Its Applications" (2016)

---

## 🔄 Quick Reference: Using Saved Model in Future Sessions

### Option 1: Re-run This Notebook for nDNA Analysis (Recommended)
```python
# In Cell 2, change:
TRAIN_NEW_MODEL = False  # ← This skips training and loads saved model

# Make sure LOCAL_BASE_DIR points to where you saved the model:
LOCAL_BASE_DIR = "E:/nDNA/30Nov2025/FinetunedModels"  # Your saved model location

# Then run: Cells 1-4, skip 5-9, run 10 onwards
# All nDNA analysis will use your previously fine-tuned model!
```

### Option 2: Standalone Script for Other Projects (Local Machine)
```python
# Use this code in any new notebook/script to load your fine-tuned model
# Works on your local machine without needing Google Drive!

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# ============================================================================
# CONFIGURE PATHS - POINT TO YOUR SAVED MODEL
# ============================================================================
# Change this to wherever you saved your fine-tuned model:
LORA_ADAPTER_PATH = "E:/nDNA/30Nov2025/FinetunedModels/latin_american_lora_adapter"
BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"

# 8-bit quantization config (must match training config)
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.float16,
    bnb_8bit_use_double_quant=True,
)

# Load tokenizer from saved adapter
tokenizer = AutoTokenizer.from_pretrained(LORA_ADAPTER_PATH)
tokenizer.pad_token = tokenizer.eos_token

# Load base model with quantization
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

# Apply your saved LoRA adapter
print("Applying LoRA adapter...")
model = PeftModel.from_pretrained(base_model, LORA_ADAPTER_PATH)
model.eval()
print("✅ Fine-tuned model loaded successfully!")

# ============================================================================
# INFERENCE FUNCTION
# ============================================================================
def generate(prompt, max_tokens=100, temperature=0.7):
    """Generate text using the fine-tuned Latin American culture model"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# ============================================================================
# EXAMPLE USAGE
# ============================================================================
# Test with Latin American cultural prompts
prompts = [
    "The ancient Maya civilization was known for",
    "Frida Kahlo's artistic style",
    "The carnival in Rio de Janeiro",
    "Traditional Mexican cuisine includes",
]

for prompt in prompts:
    print(f"\n📝 Prompt: {prompt}")
    response = generate(prompt)
    print(f"🤖 Response: {response}")
```

### Option 3: Using with Google Colab (If you saved to Google Drive)
```python
# If you originally saved to Google Drive, mount it first:
from google.colab import drive
drive.mount('/content/drive')

# Then use the Google Drive path:
LORA_ADAPTER_PATH = "/content/drive/MyDrive/nDNA_LatinAmerican/FinetunedModels/latin_american_lora_adapter"

# Rest of the code is the same as Option 2
```

---

## 📁 Model Files Explained

After training, your `latin_american_lora_adapter/` folder contains:
- `adapter_config.json` - LoRA configuration
- `adapter_model.safetensors` - LoRA weights (small, ~50-100MB)
- `tokenizer_config.json` - Tokenizer settings
- `special_tokens_map.json` - Special token mappings
- `tokenizer.json` - Tokenizer vocabulary

**Note:** The LoRA adapter is very small (~50-100MB) compared to the full model (~6GB).
When loading, we apply the adapter to the base model, which is downloaded from HuggingFace.

---
**Notebook created for Latin American Cultural SFT + nDNA Analysis Research**